In [ ]:
!pip install transformers datasets accelerate bitsandbytes torch
!pip install peft trl
!pip install sacrebleu rouge-score
!pip install --upgrade evaluate
!pip install rouge_score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 MB 43.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 564.6/564.6 kB 14.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.8/51.8 kB 3.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.1/104.1 kB 7.0 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=945d3dff1d302954fa7ac90813fb84ccccf097fd99d0429df97317d935b3d25a
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00


In [ ]:
from collections import Counter
from datasets import Dataset, DatasetDict
from google.colab import files
from huggingface_hub import login
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from peft import LoraConfig, get_peft_model, TaskType
from rouge_score import rouge_scorer
from sacrebleu import BLEU
import seaborn as sns
import shutil
from sklearn.model_selection import train_test_split
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    TrainingArguments, Trainer,
    BitsAndBytesConfig
)

In [ ]:
uploaded = files.upload()
jsonl_file = list(uploaded.keys())[0]

Saving cbc_dataset.jsonl to cbc_dataset.jsonl


In [ ]:
hf_token = "your hugging face token"
login(token=hf_token)

In [ ]:
# Setup quantization for memory efficiency
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
# Load Mistral 7B Instruct model and tokenizer
model_name = "mistralai/Mistral-7B-Instruct-v0.1"

print("Loading Mistral 7B Instruct tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading Mistral 7B Instruct model with quantization...")
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
    trust_remote_code=True
)

Loading Mistral 7B Instruct tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/4.54G [00:00<?, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.94G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

In [ ]:
# Setup LoRA for efficient fine-tuning
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
)
model = get_peft_model(model, lora_config)

In [ ]:
def load_data_mistral_single_task(filename):
    training_data = []

    with open(filename, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                item = json.loads(line)

                output = item['output']
                if 'CORRECTION:' in output and 'FEEDBACK:' in output:
                    correction = output.split('CORRECTION:')[1].split('FEEDBACK:')[0].strip()
                    feedback = output.split('FEEDBACK:')[1].strip()

                    unified_instruction = f"<s>[INST] Correct the grammar errors in this sentence and provide educational feedback explaining the corrections: {item['input']} [/INST]"

                    unified_target = f"CORRECTION: {correction}\\n\\nFEEDBACK: {feedback}"

                    training_data.append({
                        'text': unified_instruction,
                        'target': unified_target
                    })

    return training_data

training_data = load_data_mistral_single_task(jsonl_file)
train_data, val_data = train_test_split(training_data, test_size=0.1, random_state=42)
dataset = DatasetDict({
    'train': Dataset.from_list(train_data),
    'validation': Dataset.from_list(val_data)
})

In [ ]:
def preprocess_function(examples):
    inputs = examples['text']
    targets = examples['target']

    texts = []
    for input_text, target_text in zip(inputs, targets):
        full_text = input_text + " " + target_text + tokenizer.eos_token
        texts.append(full_text)

    model_inputs = tokenizer(
        texts,
        max_length=1024,
        truncation=True,
        padding=False,
        return_tensors=None
    )

    input_only = tokenizer(
        inputs,
        max_length=1024,
        truncation=True,
        padding=False,
        return_tensors=None
    )

    labels = []
    for i, (full_ids, input_ids) in enumerate(zip(model_inputs["input_ids"], input_only["input_ids"])):
        label = [-100] * len(full_ids)
        input_len = len(input_ids)
        for j in range(input_len, len(full_ids)):
            if j < len(full_ids):
                label[j] = full_ids[j]

        labels.append(label)

    model_inputs["labels"] = labels
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

Map:   0%|          | 0/2161 [00:00<?, ? examples/s]

Map:   0%|          | 0/241 [00:00<?, ? examples/s]

In [ ]:
# Training arguments - WITH validation but batch_size=1
training_args = TrainingArguments(
    output_dir="./mistral-grammar-model",
    num_train_epochs=2,
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    learning_rate=1e-5,
    save_strategy="no",
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    remove_unused_columns=False,
    warmup_steps=50,
    weight_decay=0.01,
    max_grad_norm=1.0,
    fp16=False,
    dataloader_pin_memory=False,
    report_to=None,
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],  # Add validation back
    tokenizer=tokenizer,
)

/tmp/ipython-input-2723713921.py:1: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [ ]:
model.train()
trainer.train()

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)


<IPython.core.display.Javascript object>

wandb: Logging into wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: You can find your API key in your browser here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

 ··········


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: e-obolo (j-chemirmir-glasgow-caledonian-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss,Validation Loss
100,0.554500,0.519535
200,0.408800,0.431039
300,0.466700,0.393170
400,0.322800,0.375734
500,0.339800,0.358259
600,0.394400,0.363948
700,0.405200,0.348606
800,0.267800,0.336829
900,0.299600,0.330639
1000,0.364900,0.323653


TrainOutput(global_step=4322, training_loss=0.2923897670367315, metrics={'train_runtime': 3883.5329, 'train_samples_per_second': 1.113, 'train_steps_per_second': 1.113, 'total_flos': 2.54532535197696e+16, 'train_loss': 0.2923897670367315, 'epoch': 2.0})

In [ ]:
def create_alternative_instruction():
    return """<s>[INST] Correct the grammar errors in this sentence and provide educational feedback explaining the corrections: """

def test_alternative_model(text):
    instruction_template = create_alternative_instruction()
    prompt = f"{instruction_template}{text} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt", max_length=1024, truncation=True)

    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=500,
            num_beams=4,
            early_stopping=False,
            do_sample=False,
            temperature=0.7
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "[/INST]" in result:
        result = result.split("[/INST]")[1].strip()

    # Parse the unified response
    if "CORRECTION:" in result and "FEEDBACK:" in result:
        correction = result.split("CORRECTION:")[1].split("FEEDBACK:")[0].strip()
        feedback = result.split("FEEDBACK:")[1].strip()
        return correction, feedback
    else:
        return result, "Could not parse response"

test_sentences = [
    "I has a apple.",
    "She go to school yesterday.",
    "The students was very happy.",
    "He don't like apples.",
    "We was playing football.",
    "I go to school every day.",
    "The cat run very fast in garden."
]

print("=== TESTING ALTERNATIVE MODEL (Different Instruction Structure) ===")
print("Instruction: 'Correct the grammar errors in this sentence and provide educational feedback explaining the corrections:'")
print("="*80)

for i, sentence in enumerate(test_sentences, 1):
    print(f"\\n[TEST {i}]")
    print(f"Input: {sentence}")

    correction, feedback = test_alternative_model(sentence)

    print(f"CORRECTION: {correction}")
    print(f"FEEDBACK: {feedback}")
    print("-" * 60)

print("\\n=== TESTING COMPLETED ===")


The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


=== TESTING ALTERNATIVE MODEL (Different Instruction Structure) ===
Instruction: 'Correct the grammar errors in this sentence and provide educational feedback explaining the corrections:'
\n[TEST 1]
Input: I has a apple.
CORRECTION: I have an apple.\n\n
FEEDBACK: The error is phrasing and verb tense. Change "has" to "have" for the correct subject-verb agreement. Change "a apple" to "an apple" for the correct article. This helps you use phrasing and verb tenses correctly.
------------------------------------------------------------
\n[TEST 2]
Input: She go to school yesterday.
CORRECTION: She went to school yesterday.\n\n
FEEDBACK: The error is verb tense. Change "go" to "went" for the past tense. This helps you use verb tenses accurately.
------------------------------------------------------------
\n[TEST 3]
Input: The students was very happy.
CORRECTION: The students were very happy.\n\n
FEEDBACK: The error is subject-verb agreement. Change "was" to "were" for the correct subject-ver

In [ ]:
def evaluate_alternative_model(model, tokenizer, tokenized_dataset, num_samples=50):
    val_samples = [tokenized_dataset['validation'][i] for i in range(min(num_samples, len(tokenized_dataset['validation'])))]

    predictions = []
    references = []

    for i, sample in enumerate(val_samples):
        text = tokenizer.decode(sample['input_ids'], skip_special_tokens=True)
        if "[/INST]" in text:
            target_start = text.find("[/INST]") + len("[/INST]")
            target_text = text[target_start:].strip()
        else:
            valid_labels = [label for label in sample['labels'] if label != -100]
            target_text = tokenizer.decode(valid_labels, skip_special_tokens=True)

        inputs = tokenizer(text, return_tensors="pt", max_length=1024, truncation=True)
        device = next(model.parameters()).device
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=500,
                num_beams=4,
                early_stopping=False,
                do_sample=False,
                temperature=0.7,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        if "[/INST]" in prediction:
            prediction = prediction.split("[/INST]")[1].strip()

        predictions.append(prediction)
        references.append(target_text)

    def calculate_bleu(preds, refs):
        try:
            bleu = BLEU()
            score = bleu.corpus_score(preds, [refs])
            return score.score
        except Exception as e:
            return 0.0

    def calculate_rouge(preds, refs):
        try:
            rouge = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
            rouge1_scores = []
            rouge2_scores = []
            rougeL_scores = []

            for pred, ref in zip(preds, refs):
                scores = rouge.score(ref, pred)
                rouge1_scores.append(scores['rouge1'].fmeasure)
                rouge2_scores.append(scores['rouge2'].fmeasure)
                rougeL_scores.append(scores['rougeL'].fmeasure)

            return {
                'rouge1': np.mean(rouge1_scores),
                'rouge2': np.mean(rouge2_scores),
                'rougeL': np.mean(rougeL_scores)
            }
        except Exception as e:
            return {'rouge1': 0.0, 'rouge2': 0.0, 'rougeL': 0.0}

    overall_bleu = calculate_bleu(predictions, references)
    overall_rouge = calculate_rouge(predictions, references)

    return {
        'overall_bleu': overall_bleu,
        'overall_rouge': overall_rouge,
        'num_samples': len(predictions)
    }

results = evaluate_alternative_model(model, tokenizer, tokenized_dataset, num_samples=50)
results

{'overall_bleu': 100.00000000000004,
 'overall_rouge': {'rouge1': np.float64(1.0),
  'rouge2': np.float64(1.0),
  'rougeL': np.float64(1.0)},
 'num_samples': 50}

In [ ]:
model.save_pretrained("./mistral-grammar-single-task-final")
tokenizer.save_pretrained("./mistral-grammar-single-task-final")
shutil.make_archive("mistral-grammar-single-task-final", "zip", "./mistral-grammar-single-task-final")
files.download("mistral-grammar-single-task-final.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>